# Music Generator — Google Colab Setup

Two processes run side by side:
- **Magenta service** (port 8001): Python 3.7 conda env with TF1 + Magenta
- **Main API** (port 8000): System Python with AutoGen + Gemini

## Step 1: Clone repo

In [ ]:
# If private repo: !git clone https://<TOKEN>@github.com/Jefferson8868/AI_Music.git
!git clone https://github.com/Jefferson8868/AI_Music.git
%cd AI_Music

## Step 2: Install Miniconda + Python 3.7 env for Magenta

In [ ]:
%%bash
set -e
if [ ! -f /content/miniconda/bin/conda ]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
  bash /tmp/miniconda.sh -b -p /content/miniconda
  echo 'Miniconda installed'
else
  echo 'Miniconda already installed'
fi
CONDA=/content/miniconda/bin/conda
$CONDA tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>/dev/null || true
$CONDA tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r 2>/dev/null || true
echo 'Conda ToS accepted'

In [ ]:
%%bash
set -e
CONDA=/content/miniconda/bin/conda
if [ ! -d /content/miniconda/envs/magenta_env ]; then
  $CONDA create -n magenta_env python=3.7 -y -q
fi
PY=/content/miniconda/envs/magenta_env/bin/python
if [ ! -f $PY ]; then
  echo 'ERROR: Python not found, recreating...'
  $CONDA env remove -n magenta_env -y 2>/dev/null || true
  $CONDA create -n magenta_env python=3.7 -y
fi
$PY --version

In [ ]:
# Install system build deps needed by python-rtmidi (Magenta dependency)
!apt-get install -y -qq libasound2-dev libjack-dev build-essential > /dev/null 2>&1
print('System build deps installed')

# Install Magenta + web server in Python 3.7 env
!/content/miniconda/envs/magenta_env/bin/pip install -q magenta "fastapi<0.100" "uvicorn[standard]<0.24"
print('Magenta + FastAPI installed in magenta_env')

## Step 3: Download Magenta models

In [ ]:
%%bash
mkdir -p models
[ ! -f models/attention_rnn.mag ] && wget -q http://download.magenta.tensorflow.org/models/attention_rnn.mag -P models/
[ ! -f models/polyphony_rnn.mag ] && wget -q http://download.magenta.tensorflow.org/models/polyphony_rnn.mag -P models/
ls -lh models/*.mag

## Step 4: Install main app dependencies (system Python)

In [ ]:
!pip install -q autogen-core autogen-agentchat "autogen-ext[openai]"
!pip install -q fastapi "uvicorn[standard]" "pydantic>=2.0" pydantic-settings \
    httpx music21 mido python-multipart websockets loguru \
    openai anthropic google-generativeai

## Step 5: Configure Gemini API key

Add a Colab Secret named `GEMINI_API_KEY` (click the key icon in the left sidebar).  
Get a free key at https://aistudio.google.com/apikey

In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

with open('.env', 'w') as f:
    f.write(f"""MG_LLM_BACKEND=gemini
MG_LLM_MODEL=gemini-2.0-flash
MG_LLM_API_KEY={GEMINI_API_KEY}
MG_MUSIC_ENGINE=magenta
MG_MAGENTA_URL=http://localhost:8001
MG_HOST=0.0.0.0
MG_PORT=8000
MG_MAX_GROUP_CHAT_MESSAGES=50
MG_CRITIC_PASS_THRESHOLD=0.8
""")

print('.env configured')

## Step 6: Start Magenta service (Python 3.7)

In [ ]:
import subprocess, time, requests, shutil, os

MAGENTA_PYTHON = '/content/miniconda/envs/magenta_env/bin/python'
assert os.path.isfile(MAGENTA_PYTHON), f'Not found: {MAGENTA_PYTHON}'

shutil.copy('src/engine/magenta_service.py', '/content/magenta_app.py')

magenta_log = open('/content/magenta.log', 'w')
magenta_proc = subprocess.Popen(
    [MAGENTA_PYTHON, '-m', 'uvicorn', 'magenta_app:app',
     '--host', '0.0.0.0', '--port', '8001'],
    cwd='/content',
    env={**os.environ,
         'MELODY_MODEL': '/content/AI_Music/models/attention_rnn.mag',
         'POLY_MODEL': '/content/AI_Music/models/polyphony_rnn.mag'},
    stdout=magenta_log, stderr=subprocess.STDOUT
)

print(f'Magenta started (pid={magenta_proc.pid})')
for i in range(30):
    time.sleep(2)
    if magenta_proc.poll() is not None:
        print(f'Magenta exited with code {magenta_proc.returncode}')
        with open('/content/magenta.log') as f:
            print(f.read()[-3000:])
        break
    try:
        r = requests.get('http://localhost:8001/health', timeout=3)
        print(f'Magenta ready: {r.json()}')
        break
    except:
        print(f'  waiting... ({(i+1)*2}s)', end='\r')
else:
    print('Magenta timed out. Log:')
    with open('/content/magenta.log') as f:
        print(f.read()[-3000:])

### (Debug) Check Magenta log

In [ ]:
with open('/content/magenta.log') as f:
    print(f.read()[-5000:])
print(f'Process alive: {magenta_proc.poll() is None}')

## Step 7: Start main API server

In [ ]:
import subprocess, time, requests

server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'src.main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/AI_Music',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

for i in range(15):
    time.sleep(1)
    try:
        r = requests.get('http://localhost:8000/api/health', timeout=2)
        print(f'Main server ready: {r.json()}')
        break
    except:
        print(f'  waiting... ({i+1}s)', end='\r')
else:
    print('Main server failed:')
    print(server.stdout.read1(4096).decode())

## Step 8: Generate music

In [ ]:
import requests, json

print('Generating music (1-3 minutes)...\n')

response = requests.post('http://localhost:8000/api/generate', json={
    'request': {
        'description': 'A gentle Chinese-style piano piece with pentatonic melody',
        'genre': 'classical',
        'mood': 'peaceful',
        'tempo': 80,
        'key': 'C',
        'scale_type': 'chinese_pentatonic',
        'sections': ['intro', 'verse', 'chorus', 'outro'],
        'bars_per_section': 4
    }
}, timeout=300)

result = response.json()
print(f"Status: {result['status']}")
print(f"MIDI: {result.get('midi_path', 'none')}")
print(f"Summary: {json.dumps(result.get('summary', {}), indent=2)}")
print(f"\n--- Agent Messages ({len(result.get('messages', []))}) ---")
for i, msg in enumerate(result.get('messages', [])):
    preview = msg['content'][:200].replace('\n', ' ')
    print(f"\n[{i+1}] {msg['source']}: {preview}...")

## Step 9: Play + download MIDI

In [ ]:
import glob, os
from IPython.display import Audio, display

midi_files = sorted(glob.glob('output/*.mid') + glob.glob('output/**/*.mid'),
                    key=os.path.getmtime, reverse=True)

if midi_files:
    midi_path = midi_files[0]
    print(f'Latest MIDI: {midi_path}')
    !apt-get install -y -qq fluidsynth fluid-soundfont-gm > /dev/null 2>&1
    wav_path = midi_path.replace('.mid', '.wav')
    !fluidsynth -ni /usr/share/sounds/sf2/FluidR3_GM.sf2 "{midi_path}" -F "{wav_path}" -r 44100 > /dev/null 2>&1
    if os.path.exists(wav_path):
        display(Audio(wav_path))
    else:
        print('FluidSynth failed - download MIDI directly.')
else:
    print('No MIDI files. Check agent messages above.')

In [ ]:
from google.colab import files
if midi_files:
    files.download(midi_files[0])

## Cleanup

In [ ]:
magenta_proc.terminate()
server.terminate()
print('Servers stopped.')